In [ ]:
import os
import json
from anthropic import Anthropic
import mysql.connector
import pandas as pd


# different enviorment but know this is a security issue
os.environ["ANTHROPIC_API_KEY"] = "api_key"

client = Anthropic()  # ✅ Reads from environment (Claude's original)

def ask_claude(prompt, system="You are a senior data scientist."):
    """Send a prompt to Claude and return the text response."""
    message = client.messages.create(
        model="claude-sonnet-4-6",  # ✅ Claude's original model (live now!)
        max_tokens=2048,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text

print("Ready! Run your prompt cell now.")

# Verify it worked
print("API key now visible:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("Client ready!")

Ready! Run your prompt cell now.
API key now visible: True
Client ready!


In [11]:
import mysql.connector

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_PASSWORD",
    "database": "lowes_db",
}

def run_query(sql):
    conn = mysql.connector.connect(**DB_CONFIG)
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

df = run_query("SELECT * FROM clean_reviews")

# Build a summary of the dataset for Claude
data_summary = {
    "dataset": "Lowe's NC Store Reviews",
    "rows": len(df),
    "columns": list(df.columns),
    "dtypes": {col: str(df[col].dtype) for col in df.columns},
    "numeric_stats": json.loads(
        df.describe().round(2).to_json()
    ),
    "sample_values": {
        "nc_region": df["nc_region"].value_counts().head(5).to_dict(),
        "rating_tier": df["rating_tier"].value_counts().to_dict(),
        "volume_tier": df["volume_tier"].value_counts().to_dict(),
    },
    "target_variable": "review_count",
    "business_goal": "Predict which stores will have high customer engagement (review volume)"
}

print(json.dumps(data_summary, indent=2))

{
  "dataset": "Lowe's NC Store Reviews",
  "rows": 118,
  "columns": [
    "id",
    "store_number",
    "store_name",
    "store_address",
    "city",
    "state_code",
    "zip_code",
    "latitude",
    "longitude",
    "overall_rating",
    "review_count",
    "rating_tier",
    "volume_tier",
    "nc_region"
  ],
  "dtypes": {
    "id": "int64",
    "store_number": "object",
    "store_name": "object",
    "store_address": "object",
    "city": "object",
    "state_code": "object",
    "zip_code": "object",
    "latitude": "float64",
    "longitude": "float64",
    "overall_rating": "float64",
    "review_count": "int64",
    "rating_tier": "object",
    "volume_tier": "object",
    "nc_region": "object"
  },
  "numeric_stats": {
    "id": {
      "count": 118.0,
      "mean": 59.5,
      "std": 34.21,
      "min": 1.0,
      "25%": 30.25,
      "50%": 59.5,
      "75%": 88.75,
      "max": 118.0
    },
    "latitude": {
      "count": 118.0,
      "mean": 35.52,
      "std": 0.5

/var/folders/32/xg3kzz254ng1l1tfj5yxt7k40000gn/T/ipykernel_49101/1445512103.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


In [6]:
# Have to pay for Anthropic API subscription or credits to be able to use this
prompt = f"""
I'm a data science student building a regression model to predict
review_count (customer engagement proxy) for Lowe's stores in North Carolina.

Here is my dataset summary:
{json.dumps(data_summary, indent=2)}

Based on this data, please:
1. Which features should I include in my model and why?
2. Which features should I drop and why?
3. What 2-3 new features could I engineer from the existing columns?
4. What model would you recommend for a first-year student?

Keep your answer concise and practical.
"""

response = ask_claude(prompt)
print(response)

# Regression Model Advice: Lowe's NC Review Count Prediction

## 1. Features to Include

| Feature | Reason |
|---|---|
| `overall_rating` | Direct signal — rating visibility likely drives review behavior |
| `nc_region` | Captures population density differences (Charlotte Metro vs. Eastern NC) |
| `latitude` | Geographic proxy for urban/rural split within NC |
| `rating_tier` | Encoded version of rating with business meaning |

> **Note:** Use `nc_region` OR `latitude`/`longitude` — not both, to avoid redundancy.

---

## 2. Features to Drop

| Feature | Reason |
|---|---|
| `id`, `store_number` | Arbitrary identifiers — no predictive signal |
| `store_name`, `store_address` | Too specific, causes overfitting with 118 rows |
| `state_code` | Zero variance — all rows are NC |
| `zip_code`, `city` | Too granular for 118 rows; use `nc_region` instead |
| `longitude` | **Data quality issue** — max is +80.1 (positive = wrong hemisphere), std of 25 is suspicious. Flag and investigate before

In [7]:
with open("outputs/claude_feature_selection.md", "w") as f:
    f.write("# Claude API — Feature Selection Recommendations\n\n")
    f.write(f"**Model:** claude-sonnet-4-6\n")
    f.write(f"**Date:** {pd.Timestamp.now().strftime('%Y-%m-%d')}\n\n")
    f.write("## Prompt Summary\n")
    f.write("Sent dataset schema, summary stats, and business goal.\n\n")
    f.write("## Response\n\n")
    f.write(response)

print("✓ Saved to outputs/claude_feature_selection.md")

✓ Saved to outputs/claude_feature_selection.md


In [ ]:
# features for feature_cols

# 1. is_urban flag
urban_regions = ["Charlotte Metro", "Triangle", "Triad"]
df["is_urban"] = df["nc_region"].isin(urban_regions).astype(int)

# 2. rating × urban interaction
df["rating_x_urban"] = df["overall_rating"] * df["is_urban"]

# 3. rating deviation from mean
df["rating_deviation"] = df["overall_rating"] - df["overall_rating"].mean()

X = df[["overall_rating", "is_urban", "rating_x_urban", "rating_deviation", "nc_region"]]

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.cluster import KMeans
import pandas as pd

# Select features (you can tweak)
feature_cols = [
    "overall_rating",
    "is_urban",
    "rating_x_urban",
    "rating_deviation",
    "nc_region",
]

# X and y (log‑transform review_count if needed)
X = df[feature_cols]
y = df["review_count"]  # or np.log(df["review_count"])

# One‑hot encode nc_region
X = pd.get_dummies(X, columns=["nc_region"], drop_first=True)

# Small dataset ⇒ keep test size limited
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Fit model
linear_regression = LinearRegression()
linear_regression.fit(X_train, y_train)

# Predict
y_pred = linear_regression.predict(X_test)

# Now these work
r2 = linear_regression.score(X_test, y_test)
rmse = root_mean_squared_error(y_test, y_pred)

K = 4  # or optimal K from elbow plot
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
X_cluster = df[["overall_rating", "latitude", "longitude"]]  # simple clustering features
df["cluster"] = kmeans.fit_predict(X_cluster)
profiles = df.groupby("cluster")[["review_count", "overall_rating"]].mean()

# Extract feature names and coefficients
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": linear_regression.coef_
}).sort_values(by="coef", key=abs, ascending=False)

In [28]:
# Gather your model results (run after Model 1 and Model 2 above)
model_results = {
    "linear_regression": {
        "target": "review_count",
        "R2": round(r2, 3),
        "RMSE": round(rmse, 0),
        "top_features": coef_df.head(5).to_dict(orient="records"),
    },
    "clustering": {
        "n_clusters": K,
        "cluster_profiles": profiles.to_dict(orient="index"),
    },
    "financial_highlights": {
        "latest_revenue": "$86.3B",
        "operating_margin": "11.8%",
        "store_count": 1759,
        "nc_stores_analyzed": 117,
        "avg_nc_rating": round(df["overall_rating"].mean(), 2),
    }
}

narrative_prompt = f"""
You are writing a 3-paragraph executive summary for a data science
portfolio project analyzing Lowe's Companies (NYSE: LOW).

The audience is a hiring manager or interviewer who has 30 seconds
to scan this. Make it clear, concrete, and impressive.

Here are the analysis results:
{json.dumps(model_results, indent=2)}

Write the summary covering:
1. What was analyzed and why it matters
2. Key findings from the models
3. Business recommendations

Use plain English. No jargon. Include specific numbers.
"""

narrative = ask_claude(
    narrative_prompt,
    system="You are a business analytics consultant writing for executives."
)
print(narrative)

## Executive Summary: Lowe's Companies (NYSE: LOW) — Store Performance Analysis

This project analyzed 117 Lowe's store locations across North Carolina, set against the backdrop of Lowe's $86.3 billion in annual revenue and 1,759 stores nationwide. North Carolina is Lowe's home state and a strategically significant market, making it an ideal lens for understanding what drives customer engagement at the store level. The analysis combined customer review data, geographic segmentation, and financial context to answer a practical question: which store characteristics predict higher customer engagement, and where are performance gaps hiding?

The models surfaced several concrete findings. A linear regression predicting review volume (a proxy for customer engagement) identified urban location as the single strongest driver — urban stores attract roughly 1,365 more reviews on average than rural counterparts. Interestingly, stores in the Mountains/Asheville region underperform by approximately

In [29]:
with open("outputs/claude_executive_summary.md", "w") as f:
    f.write("# Executive Summary — Lowe's NC Analytics Project\n\n")
    f.write(f"*Generated by Claude API (claude-sonnet-4-6) on "
            f"{pd.Timestamp.now().strftime('%Y-%m-%d')}*\n\n")
    f.write(narrative)

print("✓ Saved to outputs/claude_executive_summary.md")
print("  → Use this in your Notion portfolio page and GitHub README")

✓ Saved to outputs/claude_executive_summary.md
  → Use this in your Notion portfolio page and GitHub README
